# Redrob Candidate Ranking — Sandbox Demo

Reproduces the **ranking step** on a small candidate sample, CPU-only, under 5 minutes.

Clones the public repo, installs dependencies, builds a 100-candidate sample, runs `rank.py`, and shows the ranked output.

The full 100K reproduction happens in the organizers' Stage-3 environment; this is the small-sample sanity check required by submission_spec Section 10.5.

## 1. Clone the repo
Replace the URL with your public repo.

In [ ]:
import os, shutil, glob
shutil.rmtree('redrob', ignore_errors=True)
!git clone https://github.com/USERNAME/REPO.git redrob
# cd into the folder that has both rank.py and artifacts/
target = None
for root, dirs, fnames in os.walk('redrob'):
    if 'rank.py' in fnames and 'artifacts' in dirs:
        target = root; break
assert target, 'rank.py + artifacts/ not found — check the repo URL and that artifacts are committed'
os.chdir(target)
print('working dir:', os.getcwd())
print('parquet present:', os.path.isfile('artifacts/features_100k.parquet'))

## 2. Install dependencies (CPU)
Use Colab's existing numpy/pandas; only add lightgbm + pyarrow to avoid version clashes.

In [ ]:
!pip install -q lightgbm pyarrow

## 3. Build a 100-candidate sample

In [ ]:
import json, pandas as pd
df = pd.read_parquet('artifacts/features_100k.parquet')
sample_ids = list(df.index[:100])
try:
    full = {c['candidate_id']: c for c in json.load(open('sample_candidates.json'))}
except Exception:
    full = {}
with open('sandbox_candidates.jsonl', 'w') as f:
    for cid in sample_ids:
        rec = full.get(cid, {'candidate_id': cid, 'profile': {}, 'career_history': [], 'skills': [], 'redrob_signals': {}})
        f.write(json.dumps(rec) + '\n')
print('wrote', len(sample_ids), 'candidates')

## 4. Run the ranking step (CPU, < 5 min)

In [ ]:
import time
t0 = time.time()
!python rank.py --candidates sandbox_candidates.jsonl --out sandbox_submission.csv
print(f'\nelapsed: {time.time()-t0:.1f}s')

## 5. Show the ranked output

In [ ]:
import pandas as pd
out = pd.read_csv('sandbox_submission.csv')
print('rows:', len(out), '| distinct reasoning:', out.reasoning.nunique())
out.head(15)